# Load model data

In [79]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = "llama-3.1-8b-instant"
GEMINI_API_KEY = os.getenv("GEMINI_API")
GEMINI_MODEL = "gemma-3-4b-it"

In [80]:
# Constants and configuration
CHUNK_SIZE = 25
OUTPUT_FOLDER = "../data/scored_data"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [81]:
import re
import numpy as np
import pandas as pd

def parse_scores(response_text, expected_count):
    """
    Extracts all digits/floats from model response, 
    rounds answers to the nearest integer, and clamps them to 1-5.
    Returns exactly 'expected_count' results (pad with None if missing).
    """
    if not response_text:
        return [pd.NA] * expected_count
        
    # Find sequence of numbers
    matches = re.findall(r"\b\d+(?:\.\d+)?\b", str(response_text))
    
    scores = []
    for match in matches:
        try:
            val = float(match)
            val = int(round(val))
            val = max(1, min(5, val))
            scores.append(val)
        except ValueError:
            pass
            
    # Normalize length
    if len(scores) < expected_count:
        scores.extend([pd.NA] * (expected_count - len(scores)))
    elif len(scores) > expected_count:
        scores = scores[:expected_count]
        
    return scores

def build_prompt(reviews_chunk):
    prompt = (
        "Analyze the sentiment of the following product reviews. "
        "For each review, provide a single integer score from 1 to 5, "
        "where 1 is very negative and 5 is very positive. No decimals, no halves. "
        "Provide ONLY the numeric scores separated by commas in the exact same order as the reviews. "
        "Do not include any text, explanations, or headings.\n\n"
    )
    for i, review in enumerate(reviews_chunk):
        # Trim excessively long reviews to preserve input tokens (e.g. max 1000 chars)
        trimmed_review = str(review)[:1000]
        prompt += f"Review {i+1}:\n{trimmed_review}\n\n"
    return prompt

# Load datasets

In [82]:
import pandas as pd

all_cleaned_df = pd.read_csv("../data/cleaned_data/cleaned_reviews_all_flags.csv")
no_lemma_no_stopwords_df = pd.read_csv("../data/cleaned_data/cleaned_reviews_no_lemma_no_stopwords.csv")
no_special_no_lowercase_df = pd.read_csv("../data/cleaned_data/cleaned_reviews_no_speical_no_lowercase.csv")

# Get ground truth from models

In [83]:
import time
from groq import Groq
from google import genai
from google.genai import types

# Setup clients
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
gemini_client = genai.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY else None

def get_groq_scores(reviews_chunk):
    if not groq_client:
        return [pd.NA] * len(reviews_chunk)
        
    prompt = build_prompt(reviews_chunk)
    
    for attempt in range(3):
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0
            )
            result_text = response.choices[0].message.content
            scores = parse_scores(result_text, len(reviews_chunk))
            
            # If all are NA, try again; otherwise success
            if not all(pd.isna(x) for x in scores):
                return scores
        except Exception as e:
            err_msg = str(e).lower()
            if "429" in err_msg or "rate limit" in err_msg:
                print(f"Groq Rate Limit (429) hit. Waiting 60 seconds... (Attempt {attempt+1})")
                time.sleep(60)
            else:
                print(f"Groq error: {e}. Waiting 2s... (Attempt {attempt+1})")
                time.sleep(2)
            
    return [pd.NA] * len(reviews_chunk)

def get_gemini_scores(reviews_chunk):
    if not gemini_client:
        return [pd.NA] * len(reviews_chunk)
        
    prompt = build_prompt(reviews_chunk)
    for attempt in range(3):
        try:
            response = gemini_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0
                )
            )
            scores = parse_scores(response.text, len(reviews_chunk))
            
            if not all(pd.isna(x) for x in scores):
                return scores
        except Exception as e:
            err_msg = str(e).lower()
            if "429" in err_msg or "rate limit" in err_msg:
                print(f"Gemini Rate Limit (429) hit. Waiting 60 seconds... (Attempt {attempt+1})")
                time.sleep(60)
            else:
                print(f"Gemini error: {e}. Waiting 2s... (Attempt {attempt+1})")
                time.sleep(2)
            
    return [pd.NA] * len(reviews_chunk)

In [84]:
from tqdm.auto import tqdm
import os

# Registry of datasets to process
datasets_to_run = {
    "all_flags": ("cleaned_reviews_all_flags.csv", all_cleaned_df),
    "no_lemma_no_stopwords": ("cleaned_reviews_no_lemma_no_stopwords.csv", no_lemma_no_stopwords_df),
    "no_special_no_lowercase": ("cleaned_reviews_no_speical_no_lowercase.csv", no_special_no_lowercase_df)
}

In [85]:
# EXECUTE ALL DATASETS
for label, (filename, df) in datasets_to_run.items():
    print(f"\n--- Processing dataset: {label} ---")
    
    # Check if this df was indeed loaded
    if df is None or df.empty:
        print(f"Skipping {label} because it's empty or not loaded.")
        continue
        
    df_scored = df.copy().reset_index(drop=True)
    
    # Derive output filename
    base_name, ext = os.path.splitext(filename)
    out_filename = f"{base_name}_scored{ext}"
    out_path = os.path.join(OUTPUT_FOLDER, out_filename)
        
    text_items = (df_scored['review_title'].fillna('') + " \n " + df_scored['review_body'].fillna('')).tolist()
    
    # 1. Score with Gemini first
    print(f"[{label}] Scoring with Gemini...")
    df_scored['score_gemini'] = pd.NA
    for i in tqdm(range(0, len(text_items), CHUNK_SIZE), desc="Gemini"):
        chunk = text_items[i : i + CHUNK_SIZE]
        gemini_scores = get_gemini_scores(chunk)
        df_scored.iloc[i : i + len(chunk), df_scored.columns.get_loc('score_gemini')] = gemini_scores
        # Respect 15 RPM rate limit (~4 seconds per request)
        time.sleep(4.1)
        
    # 2. Score with Groq next
    print(f"[{label}] Scoring with Groq...")
    df_scored['score_groq'] = pd.NA
    for i in tqdm(range(0, len(text_items), CHUNK_SIZE), desc="Groq"):
        chunk = text_items[i : i + CHUNK_SIZE]
        groq_scores = get_groq_scores(chunk)
        df_scored.iloc[i : i + len(chunk), df_scored.columns.get_loc('score_groq')] = groq_scores
        # Respect 30 RPM rate limit (~2 seconds per request)
        time.sleep(2.1)
        
    # Save final results for this dataset once both models are done
    df_scored.to_csv(out_path, index=False)
    print(f"[{label}] Saved Gemini and Groq progression completely to {out_path}.")
    
    # Small validation
    print(f"  Valid Gemini scores: {df_scored['score_gemini'].notna().sum()}/{len(df_scored)}")
    print(f"  Valid Groq scores: {df_scored['score_groq'].notna().sum()}/{len(df_scored)}")


--- Processing dataset: all_flags ---
[all_flags] Scoring with Gemini...



--- Processing dataset: all_flags ---
[all_flags] Scoring with Gemini...


Gemini: 100%|██████████| 17/17 [01:38<00:00,  5.79s/it]



--- Processing dataset: all_flags ---
[all_flags] Scoring with Gemini...


Gemini: 100%|██████████| 17/17 [01:38<00:00,  5.79s/it]


...

[no_lemma_no_stopwords] Saved Gemini and Groq progression completely to ../data/scored_data/cleaned_reviews_no_lemma_no_stopwords_scored.csv.
  Valid Gemini scores: 409/409
  Valid Groq scores: 403/409

--- Processing dataset: no_special_no_lowercase ---
[no_special_no_lowercase] Scoring with Gemini...


Gemini: 100%|██████████| 17/17 [01:39<00:00,  5.86s/it]


[no_special_no_lowercase] Scoring with Groq...


Groq: 100%|██████████| 17/17 [02:11<00:00,  7.71s/it]

[no_special_no_lowercase] Saved Gemini and Groq progression completely to ../data/scored_data/cleaned_reviews_no_speical_no_lowercase_scored.csv.
  Valid Gemini scores: 409/409
  Valid Groq scores: 404/409


In [86]:
# Reload and clean datasets with invalid scores
import pandas as pd
import os

# Map labels to their generated scored file paths
scored_paths = {
    label: os.path.join(OUTPUT_FOLDER, f"{os.path.splitext(filename)[0]}_scored{os.path.splitext(filename)[1]}")
    for label, (filename, _) in datasets_to_run.items()
}

# Load them
scored_dfs = {label: pd.read_csv(path) for label, path in scored_paths.items() if os.path.exists(path)}

if not scored_dfs:
    print("No scored files found. Ensure the previous cell finished successfully.")
else:
    # Identify rows with invalid scores (NaN) or missing text in ANY of the datasets
    # If a row is invalid in one, we remove it from all to keep them synced.
    indices_to_drop = set()
    
    for label, df in scored_dfs.items():
        # Invalid if Gemini or Groq failed to provide a score
        invalid_mask = df['score_gemini'].isna() | df['score_groq'].isna()
        
        # Also check for empty reviews just in case
        invalid_text = df['review_body'].isna() & df['review_title'].isna()
        
        bad_indices = df[invalid_mask | invalid_text].index
        indices_to_drop.update(bad_indices)
        print(f"[{label}] Found {len(bad_indices)} rows with invalid data/scores.")

    print(f"\nTotal unique rows to remove across all datasets: {len(indices_to_drop)}")

    # Remove and overwrite
    for label, df in scored_dfs.items():
        # Drop the indices and reset
        cleaned_df = df.drop(index=list(indices_to_drop)).reset_index(drop=True)
        
        # Save back to the same path
        out_path = scored_paths[label]
        cleaned_df.to_csv(out_path, index=False)
        print(f"[{label}] Cleaned and saved {len(cleaned_df)} rows to {out_path}.")

[all_flags] Found 6 rows with invalid data/scores.
[no_lemma_no_stopwords] Found 7 rows with invalid data/scores.
[no_special_no_lowercase] Found 6 rows with invalid data/scores.

Total unique rows to remove across all datasets: 12
[all_flags] Cleaned and saved 397 rows to ../data/scored_data/cleaned_reviews_all_flags_scored.csv.
[no_lemma_no_stopwords] Cleaned and saved 397 rows to ../data/scored_data/cleaned_reviews_no_lemma_no_stopwords_scored.csv.
[no_special_no_lowercase] Cleaned and saved 397 rows to ../data/scored_data/cleaned_reviews_no_speical_no_lowercase_scored.csv.


In [89]:
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
import numpy as np

def calculate_agreement(df, rating):
    print(f"\n--- Agreement Analysis for {rating} ---")
    
    # Helper for Fleiss' Kappa
    def get_kappa(rater_columns):
        # Extract columns and drop any remaining NaNs
        rater_matrix = df[rater_columns].copy()
        for col in rater_columns:
            rater_matrix[col] = pd.to_numeric(rater_matrix[col], errors='coerce')
        rater_matrix = rater_matrix.dropna().values
        
        if len(rater_matrix) == 0:
            return np.nan

        # Our scores are 1-5, so we subtract 1 to make them 0-4
        coded_data = (rater_matrix - 1).astype(int)
        # Aggregate into counts per category
        agg_counts, _ = aggregate_raters(coded_data)
        # Calculate Fleiss' Kappa
        return fleiss_kappa(agg_counts, method='fleiss')

    def get_interpretation(kappa):
        if np.isnan(kappa): return "No valid data"
        if kappa < 0: return "Poor agreement"
        elif kappa <= 0.20: return "Slight agreement"
        elif kappa <= 0.40: return "Fair agreement"
        elif kappa <= 0.60: return "Moderate agreement"
        elif kappa <= 0.80: return "Substantial agreement"
        else: return "Almost perfect agreement"

    # 1. Agreement between Models (Inter-Annotator)
    kappa_models = get_kappa(['score_gemini', 'score_groq'])
    print(f"[Gemini vs Groq]   Fleiss' Kappa: {kappa_models:.4f} ({get_interpretation(kappa_models)})")
    
    # 2. Agreement between Models and Ground Truth
    # Ensure ground truth 'rating' is present
    if 'rating' in df.columns:
        kappa_gemini = get_kappa(['score_gemini', 'rating'])
        print(f"[Gemini vs Actual] Fleiss' Kappa: {kappa_gemini:.4f} ({get_interpretation(kappa_gemini)})")
        
        kappa_groq = get_kappa(['score_groq', 'rating'])
        print(f"[Groq vs Actual]   Fleiss' Kappa: {kappa_groq:.4f} ({get_interpretation(kappa_groq)})")
    else:
        print("Note: 'rating' column not found for comparison with ground truth.")

# Calculate for each dataset
for rating, df in scored_dfs.items():
    calculate_agreement(df, rating)


--- Agreement Analysis for all_flags ---
[Gemini vs Groq]   Fleiss' Kappa: 0.2387 (Fair agreement)
[Gemini vs Actual] Fleiss' Kappa: 0.1853 (Slight agreement)
[Groq vs Actual]   Fleiss' Kappa: 0.4220 (Moderate agreement)

--- Agreement Analysis for no_lemma_no_stopwords ---
[Gemini vs Groq]   Fleiss' Kappa: 0.2698 (Fair agreement)
[Gemini vs Actual] Fleiss' Kappa: 0.2268 (Fair agreement)
[Groq vs Actual]   Fleiss' Kappa: 0.4730 (Moderate agreement)

--- Agreement Analysis for no_special_no_lowercase ---
[Gemini vs Groq]   Fleiss' Kappa: 0.2308 (Fair agreement)
[Gemini vs Actual] Fleiss' Kappa: 0.1822 (Slight agreement)
[Groq vs Actual]   Fleiss' Kappa: 0.4356 (Moderate agreement)
